# E-commerce Checkout A/B Test  
## Data Preparation and Simulation

### Objective

This notebook creates a realistic event-level dataset for an e-commerce checkout experiment.

The simulated data represents a user-level randomized experiment comparing:

- **Control:** Existing multi-step checkout flow
- **Treatment:** Simplified single-page checkout flow

The datasets are intentionally stored at different grains to reflect a realistic analytics environment:

- one row per user;
- one row per experiment assignment record;
- one row per behavioral event; and
- one row per order.

A small number of intentional data-quality issues are included so that they can be identified and resolved during SQL validation.

### Experiment Design

- **Assignment unit:** User
- **Primary exposure:** First valid `checkout_view` after assignment
- **Primary outcome:** At least one successful purchase within 24 hours after exposure
- **Experiment assignment window:** July 1–14, 2026
- **Expected treatment effect:** Moderate positive lift in checkout purchase conversion
- **Primary analysis grain:** One row per eligible exposed user

### Intentional Data-Quality Issues

The raw datasets may contain:

- duplicate assignment records;
- users assigned to multiple experiment groups;
- duplicate event records;
- purchases occurring outside the 24-hour attribution window;
- checkout events recorded before experiment assignment;
- employee, test-account, and bot traffic;
- users without complete conversion observation windows.

These issues should not be silently removed during data generation. They will be detected and handled explicitly during the validation and analysis stages.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RANDOM_SEED = 3072
rng = np.random.default_rng(RANDOM_SEED)

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in [START_DIR, *START_DIR.parents]
        if path.name == "week2_checkout_experiment"
    ),
    START_DIR / "week2_checkout_experiment",
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.name)
print("Raw data directory:", RAW_DATA_DIR.relative_to(PROJECT_ROOT))
print("Processed data directory:", PROCESSED_DATA_DIR.relative_to(PROJECT_ROOT))

Project root: week2_checkout_experiment
Raw data directory: data/raw
Processed data directory: data/processed


## 1. Simulation Configuration

The simulation uses 20,000 users assigned approximately evenly between the control and treatment groups.

Not every assigned user will reach checkout. This distinction allows the analysis to separate experiment assignment from actual product exposure.

The raw data will preserve users with incomplete observation windows and other data-quality issues so that eligibility rules can be applied transparently during validation.

In [2]:
N_USERS = 20_000

EXPERIMENT_NAME = "checkout_flow_test"
EXPERIMENT_START = pd.Timestamp("2026-07-01 00:00:00")
EXPERIMENT_END = pd.Timestamp("2026-07-14 23:59:59")

# Behavioral event data are assumed to be available only through this timestamp.
# Users exposed after July 14 at noon will not have a complete 24-hour window.
EVENT_DATA_CUTOFF = pd.Timestamp("2026-07-15 12:00:00")

# Order-status data have a longer follow-up period for refund/cancellation outcomes.
ORDER_DATA_CUTOFF = pd.Timestamp("2026-07-22 23:59:59")

print("Users:", f"{N_USERS:,}")
print("Experiment window:", EXPERIMENT_START, "to", EXPERIMENT_END)
print("Event data cutoff:", EVENT_DATA_CUTOFF)
print("Order data cutoff:", ORDER_DATA_CUTOFF)

Users: 20,000
Experiment window: 2026-07-01 00:00:00 to 2026-07-14 23:59:59
Event data cutoff: 2026-07-15 12:00:00
Order data cutoff: 2026-07-22 23:59:59


In [3]:
user_ids = [f"U{i:06d}" for i in range(1, N_USERS + 1)]

is_new_user = rng.random(N_USERS) < 0.35

new_first_seen_seconds = rng.integers(
    0,
    int((EXPERIMENT_END - EXPERIMENT_START).total_seconds()) + 1,
    size=N_USERS,
)

returning_window_start = pd.Timestamp("2025-07-01")
returning_first_seen_seconds = rng.integers(
    0,
    int((EXPERIMENT_START - returning_window_start).total_seconds()),
    size=N_USERS,
)

first_seen_timestamp = np.where(
    is_new_user,
    EXPERIMENT_START + pd.to_timedelta(new_first_seen_seconds, unit="s"),
    returning_window_start + pd.to_timedelta(returning_first_seen_seconds, unit="s"),
)

users = pd.DataFrame(
    {
        "user_id": user_ids,
        "first_seen_timestamp": pd.to_datetime(first_seen_timestamp),
        "user_type": np.where(is_new_user, "new", "returning"),
        "is_employee": rng.random(N_USERS) < 0.003,
        "is_test_account": rng.random(N_USERS) < 0.002,
        "is_bot": rng.random(N_USERS) < 0.005,
    }
)

users.head()

,user_id,first_seen_timestamp,user_type,is_employee,is_test_account,is_bot
0,U000001,2025-09-08 07:12:07,returning,False,False,False
1,U000002,2025-12-28 01:13:41,returning,False,False,False
2,U000003,2026-07-10 00:46:35,new,False,False,False
3,U000004,2025-10-14 23:14:42,returning,False,False,False
4,U000005,2026-07-11 23:13:55,new,False,False,False


In [4]:
print("Rows:", len(users))
print("Unique users:", users["user_id"].nunique())
print()
print(users["user_type"].value_counts(normalize=True).round(4))
print()
print(users[["is_employee", "is_test_account", "is_bot"]].sum())

Rows: 20000
Unique users: 20000

user_type
returning    0.6476
new          0.3524
Name: proportion, dtype: float64

is_employee         51
is_test_account     43
is_bot             116
dtype: int64


## 2. Experiment Assignments

Users are assigned approximately evenly to the control and treatment groups.

The main assignment table initially contains one assignment per user. A small number of duplicate and conflicting assignment records are added intentionally to validate assignment-integrity controls.

The presence of multiple assignment records does not automatically mean that a user experienced both product versions. Assignment integrity and exposure integrity must be evaluated separately.

In [5]:
# Generate chronologically valid experiment assignments.
#
# Returning users can be assigned any time after the experiment starts.
# New users can only be assigned after they are first observed on the platform.

assignment_start = users["first_seen_timestamp"].copy()

assignment_start = assignment_start.where(
    assignment_start > EXPERIMENT_START,
    EXPERIMENT_START,
)

available_seconds = (
    EXPERIMENT_END - assignment_start
).dt.total_seconds().astype(int)

assignment_offset_seconds = np.array(
    [
        rng.integers(0, seconds + 1)
        for seconds in available_seconds
    ]
)

assignment_timestamp = (
    assignment_start
    + pd.to_timedelta(assignment_offset_seconds, unit="s")
)

assignments_clean = pd.DataFrame(
    {
        "assignment_id": [
            f"A{i:06d}" for i in range(1, N_USERS + 1)
        ],
        "experiment_name": EXPERIMENT_NAME,
        "user_id": user_ids,
        "experiment_group": rng.choice(
            ["control", "treatment"],
            size=N_USERS,
            p=[0.50, 0.50],
        ),
        "assignment_timestamp": assignment_timestamp,
    }
)

assignments_clean.head()

,assignment_id,experiment_name,user_id,experiment_group,assignment_timestamp
0,A000001,checkout_flow_test,U000001,control,2026-07-10 01:48:57
1,A000002,checkout_flow_test,U000002,treatment,2026-07-08 17:33:58
2,A000003,checkout_flow_test,U000003,control,2026-07-12 10:15:32
3,A000004,checkout_flow_test,U000004,treatment,2026-07-09 12:50:32
4,A000005,checkout_flow_test,U000005,control,2026-07-14 05:39:05


In [6]:
# Validate assignment chronology.

assignment_timing_check = (
    assignments_clean[
        ["user_id", "assignment_timestamp"]
    ]
    .merge(
        users[
            ["user_id", "first_seen_timestamp"]
        ],
        on="user_id",
        how="left",
        validate="one_to_one",
    )
)

assignment_before_first_seen = (
    assignment_timing_check["assignment_timestamp"]
    < assignment_timing_check["first_seen_timestamp"]
).sum()

print(
    "Assignments before first seen:",
    assignment_before_first_seen
)

assert assignment_before_first_seen == 0

Assignments before first seen: 0


In [7]:
assignment_summary = (
    assignments_clean.groupby("experiment_group")
    .agg(
        users=("user_id", "nunique"),
        assignment_records=("assignment_id", "count"),
    )
)

assignment_summary["share"] = (
    assignment_summary["users"] / assignment_summary["users"].sum()
)

assignment_summary

,users,assignment_records,share
experiment_group,,,
control,10022,10022,0.5011
treatment,9978,9978,0.4989


In [8]:
# Exact duplicate assignment records
duplicate_assignments = assignments_clean.sample(
    n=40,
    random_state=RANDOM_SEED,
).copy()

# Conflicting assignments: same user assigned to the opposite group
conflicting_assignments = assignments_clean.sample(
    n=20,
    random_state=RANDOM_SEED + 1,
).copy()

conflicting_assignments["assignment_id"] = [
    f"AX{i:04d}" for i in range(1, len(conflicting_assignments) + 1)
]

conflicting_assignments["experiment_group"] = (
    conflicting_assignments["experiment_group"]
    .map({"control": "treatment", "treatment": "control"})
)

conflicting_assignments["assignment_timestamp"] = (
    conflicting_assignments["assignment_timestamp"]
    + pd.to_timedelta(
        rng.integers(
            60,
            3600,
            size=len(conflicting_assignments),
        ),
        unit="s",
    )
)

conflicting_assignments["assignment_timestamp"] = (
    conflicting_assignments["assignment_timestamp"]
    .clip(upper=EXPERIMENT_END)
)

experiment_assignments = pd.concat(
    [
        assignments_clean,
        duplicate_assignments,
        conflicting_assignments,
    ],
    ignore_index=True,
)

experiment_assignments = experiment_assignments.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(drop=True)

print("Clean assignment rows:", len(assignments_clean))
print("Raw assignment rows:", len(experiment_assignments))
print("Unique users:", experiment_assignments["user_id"].nunique())

Clean assignment rows: 20000
Raw assignment rows: 20060
Unique users: 20000


In [9]:
# Final assignment sanity checks.

assignment_sanity = (
    experiment_assignments
    .merge(
        users[
            ["user_id", "first_seen_timestamp"]
        ],
        on="user_id",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Assignments before first seen:",
    (
        assignment_sanity["assignment_timestamp"]
        < assignment_sanity["first_seen_timestamp"]
    ).sum()
)

print(
    "Assignments before experiment start:",
    (
        assignment_sanity["assignment_timestamp"]
        < EXPERIMENT_START
    ).sum()
)

print(
    "Assignments after experiment end:",
    (
        assignment_sanity["assignment_timestamp"]
        > EXPERIMENT_END
    ).sum()
)

Assignments before first seen: 0
Assignments before experiment start: 0
Assignments after experiment end: 0


In [10]:
exposure_base = (
    assignments_clean[
        [
            "user_id",
            "experiment_group",
            "assignment_timestamp",
        ]
    ]
    .merge(
        users[
            [
                "user_id",
                "user_type",
                "is_employee",
                "is_test_account",
                "is_bot",
            ]
        ],
        on="user_id",
        how="left",
        validate="one_to_one",
    )
)

exposure_probability = np.where(
    exposure_base["user_type"].eq("returning"),
    0.88,
    0.82,
)

exposure_base["became_exposed"] = (
    rng.random(len(exposure_base)) < exposure_probability
)

exposed_users = exposure_base.loc[
    exposure_base["became_exposed"]
].copy()

exposure_delay_minutes = rng.gamma(
    shape=2.0,
    scale=90.0,
    size=len(exposed_users),
)

exposed_users["exposure_timestamp"] = (
    exposed_users["assignment_timestamp"]
    + pd.to_timedelta(exposure_delay_minutes, unit="m")
)

# Ensure the main exposure remains reasonably close to the experiment window.
exposed_users["exposure_timestamp"] = exposed_users[
    "exposure_timestamp"
].clip(upper=EXPERIMENT_END)

exposed_users["device_at_exposure"] = rng.choice(
    ["mobile", "desktop", "tablet"],
    size=len(exposed_users),
    p=[0.58, 0.36, 0.06],
)

exposed_users["traffic_source_at_exposure"] = rng.choice(
    ["organic", "paid_search", "email", "social", "direct"],
    size=len(exposed_users),
    p=[0.30, 0.25, 0.16, 0.12, 0.17],
)

print("Assigned users:", f"{len(exposure_base):,}")
print("Exposed users:", f"{len(exposed_users):,}")
print("Exposure rate:", round(len(exposed_users) / len(exposure_base), 4))

Assigned users: 20,000
Exposed users: 17,193
Exposure rate: 0.8597


In [11]:
purchase_probability = np.full(len(exposed_users), 0.39)

purchase_probability += np.where(
    exposed_users["experiment_group"].eq("treatment"),
    0.015,
    0.0,
)

purchase_probability += exposed_users["device_at_exposure"].map(
    {
        "desktop": 0.018,
        "mobile": -0.008,
        "tablet": -0.015,
    }
).to_numpy()

purchase_probability += exposed_users["traffic_source_at_exposure"].map(
    {
        "organic": 0.000,
        "paid_search": -0.006,
        "email": 0.025,
        "social": -0.015,
        "direct": 0.012,
    }
).to_numpy()

purchase_probability += np.where(
    exposed_users["user_type"].eq("returning"),
    0.020,
    -0.010,
)

purchase_probability = np.clip(
    purchase_probability,
    0.05,
    0.80,
)

exposed_users["will_purchase"] = (
    rng.random(len(exposed_users)) < purchase_probability
)

exposed_users.groupby("experiment_group")["will_purchase"].mean()

experiment_group
control      0.407025
treatment    0.416248
Name: will_purchase, dtype: float64

## 3. Behavioral Events

The event table is stored at the event level. A single user may generate multiple rows.

The expected checkout journey is:

```text
checkout_view
→ payment_attempt
→ purchase or payment_failure

In [12]:
event_records = []
event_counter = 1

def add_event(
    user_id,
    event_timestamp,
    event_name,
    experiment_group,
    device,
    traffic_source,
    session_id,
):
    global event_counter

    event_records.append(
        {
            "event_id": f"E{event_counter:08d}",
            "user_id": user_id,
            "event_timestamp": pd.Timestamp(event_timestamp),
            "event_name": event_name,
            "experiment_group": experiment_group,
            "device": device,
            "traffic_source": traffic_source,
            "session_id": session_id,
        }
    )

    event_counter += 1

In [13]:
order_inputs = []

for row in exposed_users.itertuples(index=False):
    session_id = f"S_{row.user_id}_{int(row.exposure_timestamp.timestamp())}"

    add_event(
        user_id=row.user_id,
        event_timestamp=row.exposure_timestamp,
        event_name="checkout_view",
        experiment_group=row.experiment_group,
        device=row.device_at_exposure,
        traffic_source=row.traffic_source_at_exposure,
        session_id=session_id,
    )

    # Checkout error probability is slightly lower in treatment.
    error_probability = (
        0.035 if row.experiment_group == "control" else 0.030
    )

    if rng.random() < error_probability:
        error_time = row.exposure_timestamp + pd.to_timedelta(
            rng.integers(5, 180),
            unit="s",
        )

        add_event(
            user_id=row.user_id,
            event_timestamp=error_time,
            event_name="checkout_error",
            experiment_group=row.experiment_group,
            device=row.device_at_exposure,
            traffic_source=row.traffic_source_at_exposure,
            session_id=session_id,
        )

    # Most exposed users attempt payment, but not all.
    attempt_probability = (
        0.76 if row.experiment_group == "control" else 0.78
    )

    attempted_payment = rng.random() < attempt_probability

    if attempted_payment:
        payment_attempt_time = row.exposure_timestamp + pd.to_timedelta(
            rng.integers(2, 25),
            unit="m",
        )

        add_event(
            user_id=row.user_id,
            event_timestamp=payment_attempt_time,
            event_name="payment_attempt",
            experiment_group=row.experiment_group,
            device=row.device_at_exposure,
            traffic_source=row.traffic_source_at_exposure,
            session_id=session_id,
        )

        payment_failed = rng.random() < (
            0.055 if row.experiment_group == "control" else 0.054
        )

        if payment_failed:
            failure_time = payment_attempt_time + pd.to_timedelta(
                rng.integers(10, 180),
                unit="s",
            )

            add_event(
                user_id=row.user_id,
                event_timestamp=failure_time,
                event_name="payment_failure",
                experiment_group=row.experiment_group,
                device=row.device_at_exposure,
                traffic_source=row.traffic_source_at_exposure,
                session_id=session_id,
            )

        if row.will_purchase and not payment_failed:
            # Most purchases occur within 24 hours.
            if rng.random() < 0.96:
                purchase_delay_minutes = rng.gamma(
                    shape=2.0,
                    scale=8.0,
                )
            else:
                # A small number occur outside the primary attribution window.
                purchase_delay_minutes = rng.uniform(
                    24 * 60 + 1,
                    72 * 60,
                )

            purchase_time = (
                payment_attempt_time
                + pd.to_timedelta(purchase_delay_minutes, unit="m")
            )

            # Event data are only available through the event cutoff.
            if purchase_time <= EVENT_DATA_CUTOFF:
                add_event(
                    user_id=row.user_id,
                    event_timestamp=purchase_time,
                    event_name="purchase",
                    experiment_group=row.experiment_group,
                    device=row.device_at_exposure,
                    traffic_source=row.traffic_source_at_exposure,
                    session_id=session_id,
                )

                order_inputs.append(
                    {
                        "user_id": row.user_id,
                        "experiment_group": row.experiment_group,
                        "purchase_timestamp": purchase_time,
                    }
                )

In [14]:
events = pd.DataFrame(event_records)

events = events.sort_values(
    ["user_id", "event_timestamp", "event_id"]
).reset_index(drop=True)

journey_timestamps = (
    events[events["event_name"].isin(["payment_attempt", "purchase"])]
    .pivot_table(
        index=["user_id", "session_id"],
        columns="event_name",
        values="event_timestamp",
        aggfunc="min",
    )
)

purchases_before_payment = (
    journey_timestamps["purchase"].notna()
    & (
        journey_timestamps["purchase"]
        < journey_timestamps["payment_attempt"]
    )
).sum()

print("Purchases before payment attempt:", purchases_before_payment)
assert purchases_before_payment == 0

events.head(10)

Purchases before payment attempt: 0


,event_id,user_id,event_timestamp,event_name,experiment_group,device,traffic_source,session_id
0,E00000001,U000001,2026-07-10 03:55:57.474342798,checkout_view,control,mobile,direct,S_U000001_1783655757
1,E00000002,U000001,2026-07-10 04:17:57.474342798,payment_attempt,control,mobile,direct,S_U000001_1783655757
2,E00000003,U000002,2026-07-08 19:53:25.247435122,checkout_view,treatment,desktop,direct,S_U000002_1783540405
3,E00000004,U000003,2026-07-12 14:02:33.743649830,checkout_view,control,mobile,direct,S_U000003_1783864953
4,E00000005,U000004,2026-07-09 15:21:40.318484834,checkout_view,treatment,desktop,paid_search,S_U000004_1783610500
5,E00000006,U000004,2026-07-09 15:25:40.318484834,payment_attempt,treatment,desktop,paid_search,S_U000004_1783610500
6,E00000007,U000005,2026-07-14 08:46:27.258509648,checkout_view,control,desktop,paid_search,S_U000005_1784018787
7,E00000008,U000005,2026-07-14 08:59:27.258509648,payment_attempt,control,desktop,paid_search,S_U000005_1784018787
8,E00000009,U000006,2026-07-05 04:21:24.685140208,checkout_view,treatment,mobile,social,S_U000006_1783225284
9,E00000010,U000006,2026-07-05 04:44:24.685140208,payment_attempt,treatment,mobile,social,S_U000006_1783225284


In [15]:
# Duplicate event records
duplicate_events = events.sample(
    n=80,
    random_state=RANDOM_SEED + 2,
).copy()

events = pd.concat(
    [events, duplicate_events],
    ignore_index=True,
)

# Add a small number of checkout events occurring before assignment.
pre_assignment_sample = assignments_clean.sample(
    n=30,
    random_state=RANDOM_SEED + 3,
)

invalid_event_records = []

for i, row in enumerate(pre_assignment_sample.itertuples(index=False), start=1):
    invalid_event_records.append(
        {
            "event_id": f"INVALID_PRE_{i:04d}",
            "user_id": row.user_id,
            "event_timestamp": (
                row.assignment_timestamp
                - pd.to_timedelta(rng.integers(5, 120), unit="m")
            ),
            "event_name": "checkout_view",
            "experiment_group": row.experiment_group,
            "device": rng.choice(["mobile", "desktop", "tablet"]),
            "traffic_source": rng.choice(
                ["organic", "paid_search", "email", "social", "direct"]
            ),
            "session_id": f"INVALID_S_{i:04d}",
        }
    )

events = pd.concat(
    [events, pd.DataFrame(invalid_event_records)],
    ignore_index=True,
)

events = events.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(drop=True)

print("Event rows:", f"{len(events):,}")
print("Unique event IDs:", f"{events['event_id'].nunique():,}")
print("Unique users in events:", f"{events['user_id'].nunique():,}")

Event rows: 36,813
Unique event IDs: 36,733
Unique users in events: 17,195


In [16]:
orders = pd.DataFrame(order_inputs)

orders["order_id"] = [
    f"O{i:07d}" for i in range(1, len(orders) + 1)
]

orders["order_amount"] = np.round(
    rng.lognormal(
        mean=np.log(85),
        sigma=0.45,
        size=len(orders),
    ),
    2,
)

refund_probability = np.where(
    orders["experiment_group"].eq("treatment"),
    0.038,
    0.036,
)

cancel_probability = np.where(
    orders["experiment_group"].eq("treatment"),
    0.021,
    0.020,
)

random_draw = rng.random(len(orders))

orders["order_status"] = np.select(
    [
        random_draw < refund_probability,
        random_draw < refund_probability + cancel_probability,
    ],
    [
        "refunded",
        "cancelled",
    ],
    default="completed",
)

follow_up_days = rng.integers(
    1,
    8,
    size=len(orders),
)

orders["status_update_timestamp"] = pd.NaT

status_mask = orders["order_status"].isin(["refunded", "cancelled"])

orders.loc[status_mask, "status_update_timestamp"] = (
    orders.loc[status_mask, "purchase_timestamp"]
    + pd.to_timedelta(follow_up_days[status_mask], unit="D")
)

# Outcomes after the order data cutoff are not yet observable.
immature_status_mask = (
    orders["status_update_timestamp"].notna()
    & (orders["status_update_timestamp"] > ORDER_DATA_CUTOFF)
)

orders.loc[immature_status_mask, "order_status"] = "completed"
orders.loc[immature_status_mask, "status_update_timestamp"] = pd.NaT

orders = orders[
    [
        "order_id",
        "user_id",
        "experiment_group",
        "purchase_timestamp",
        "order_amount",
        "order_status",
        "status_update_timestamp",
    ]
]

orders.head()

,order_id,user_id,experiment_group,purchase_timestamp,order_amount,order_status,status_update_timestamp
0,O0000001,U000011,control,2026-07-05 23:40:35.463412070,90.81,completed,NaT
1,O0000002,U000015,control,2026-07-13 01:00:00.086116716,112.83,completed,NaT
2,O0000003,U000016,treatment,2026-07-11 02:58:23.465040834,196.07,completed,NaT
3,O0000004,U000019,control,2026-07-14 18:14:41.135625038,69.58,completed,NaT
4,O0000005,U000027,control,2026-07-13 05:16:02.173205130,128.27,completed,NaT


In [17]:
print("Order rows:", f"{len(orders):,}")
print("Unique orders:", f"{orders['order_id'].nunique():,}")
print("Unique purchasers:", f"{orders['user_id'].nunique():,}")
print()
print(orders["order_status"].value_counts())

Order rows: 5,043
Unique orders: 5,043
Unique purchasers: 5,043

order_status
completed    4729
refunded      211
cancelled     103
Name: count, dtype: int64


## 4. Initial Data Audit

The purpose of this audit is not to clean the data. It is to confirm that the simulated tables reflect the intended grains and contain the planned validation challenges.

Cleaning decisions will be applied later using explicit business rules rather than silently modifying the raw data.

In [18]:
audit_summary = pd.DataFrame(
    {
        "table": [
            "users",
            "experiment_assignments",
            "events",
            "orders",
        ],
        "rows": [
            len(users),
            len(experiment_assignments),
            len(events),
            len(orders),
        ],
        "primary_entity": [
            "user_id",
            "assignment_id",
            "event_id",
            "order_id",
        ],
        "unique_primary_entities": [
            users["user_id"].nunique(),
            experiment_assignments["assignment_id"].nunique(),
            events["event_id"].nunique(),
            orders["order_id"].nunique(),
        ],
    }
)

audit_summary

,table,rows,primary_entity,unique_primary_entities
0,users,20000,user_id,20000
1,experiment_assignments,20060,assignment_id,20020
2,events,36813,event_id,36733
3,orders,5043,order_id,5043


In [19]:
for table_name, dataframe in {
    "users": users,
    "experiment_assignments": experiment_assignments,
    "events": events,
    "orders": orders,
}.items():
    print(f"\n{table_name.upper()}")
    print(dataframe.isna().sum())


USERS
user_id                 0
first_seen_timestamp    0
user_type               0
is_employee             0
is_test_account         0
is_bot                  0
dtype: int64

EXPERIMENT_ASSIGNMENTS
assignment_id           0
experiment_name         0
user_id                 0
experiment_group        0
assignment_timestamp    0
dtype: int64

EVENTS
event_id            0
user_id             0
event_timestamp     0
event_name          0
experiment_group    0
device              0
traffic_source      0
session_id          0
dtype: int64

ORDERS
order_id                      0
user_id                       0
experiment_group              0
purchase_timestamp            0
order_amount                  0
order_status                  0
status_update_timestamp    4729
dtype: int64


In [20]:
data_dictionary_records = [
    {
        "table_name": "users",
        "column_name": "user_id",
        "description": "Unique user identifier.",
        "grain_note": "One row per user.",
    },
    {
        "table_name": "users",
        "column_name": "first_seen_timestamp",
        "description": "Timestamp when the user was first observed on the platform.",
        "grain_note": "Used to define new versus returning users.",
    },
    {
        "table_name": "users",
        "column_name": "user_type",
        "description": "New or returning user status based on first_seen_timestamp.",
        "grain_note": "Must not be defined using purchase outcomes.",
    },
    {
        "table_name": "experiment_assignments",
        "column_name": "experiment_group",
        "description": "Assigned checkout experience: control or treatment.",
        "grain_note": "Raw data may contain duplicate or conflicting assignments.",
    },
    {
        "table_name": "experiment_assignments",
        "column_name": "assignment_timestamp",
        "description": "Timestamp of experiment assignment.",
        "grain_note": "Valid exposure should occur after assignment.",
    },
    {
        "table_name": "events",
        "column_name": "event_name",
        "description": "Behavioral event such as checkout_view, payment_attempt, payment_failure, checkout_error, or purchase.",
        "grain_note": "One row per raw event record.",
    },
    {
        "table_name": "events",
        "column_name": "event_timestamp",
        "description": "Timestamp when the behavioral event occurred.",
        "grain_note": "Used for ordering the funnel and attribution-window checks.",
    },
    {
        "table_name": "events",
        "column_name": "device",
        "description": "Device associated with the event.",
        "grain_note": "Primary segmentation will use device at first valid exposure.",
    },
    {
        "table_name": "events",
        "column_name": "traffic_source",
        "description": "Traffic source associated with the event session.",
        "grain_note": "Primary segmentation will use source at first valid exposure.",
    },
    {
        "table_name": "orders",
        "column_name": "order_amount",
        "description": "Gross amount associated with the successful purchase order.",
        "grain_note": "One row per order.",
    },
    {
        "table_name": "orders",
        "column_name": "order_status",
        "description": "Observed order outcome: completed, refunded, or cancelled.",
        "grain_note": "Requires sufficient post-purchase follow-up.",
    },
]

data_dictionary = pd.DataFrame(data_dictionary_records)

data_dictionary

,table_name,column_name,description,grain_note
0,users,user_id,Unique user identifier.,One row per user.
1,users,first_seen_timestamp,Timestamp when the user was first observed on ...,Used to define new versus returning users.
2,users,user_type,New or returning user status based on first_se...,Must not be defined using purchase outcomes.
3,experiment_assignments,experiment_group,Assigned checkout experience: control or treat...,Raw data may contain duplicate or conflicting ...
4,experiment_assignments,assignment_timestamp,Timestamp of experiment assignment.,Valid exposure should occur after assignment.
5,events,event_name,"Behavioral event such as checkout_view, paymen...",One row per raw event record.
6,events,event_timestamp,Timestamp when the behavioral event occurred.,Used for ordering the funnel and attribution-w...
7,events,device,Device associated with the event.,Primary segmentation will use device at first ...
8,events,traffic_source,Traffic source associated with the event session.,Primary segmentation will use source at first ...
9,orders,order_amount,Gross amount associated with the successful pu...,One row per order.


In [21]:
users.to_csv(
    RAW_DATA_DIR / "users.csv",
    index=False,
)

experiment_assignments.to_csv(
    RAW_DATA_DIR / "experiment_assignments.csv",
    index=False,
)

events.to_csv(
    RAW_DATA_DIR / "events.csv",
    index=False,
)

orders.to_csv(
    RAW_DATA_DIR / "orders.csv",
    index=False,
)

data_dictionary.to_csv(
    RAW_DATA_DIR / "data_dictionary.csv",
    index=False,
)

print("Files saved:")
for file_path in sorted(RAW_DATA_DIR.glob("*.csv")):
    print(f"- {file_path.name}")

Files saved:
- data_dictionary.csv
- events.csv
- experiment_assignments.csv
- orders.csv
- users.csv


In [22]:
saved_users = pd.read_csv(RAW_DATA_DIR / "users.csv")
saved_assignments = pd.read_csv(
    RAW_DATA_DIR / "experiment_assignments.csv"
)
saved_events = pd.read_csv(RAW_DATA_DIR / "events.csv")
saved_orders = pd.read_csv(RAW_DATA_DIR / "orders.csv")

validation_summary = pd.DataFrame(
    {
        "table": [
            "users",
            "experiment_assignments",
            "events",
            "orders",
        ],
        "in_memory_rows": [
            len(users),
            len(experiment_assignments),
            len(events),
            len(orders),
        ],
        "saved_file_rows": [
            len(saved_users),
            len(saved_assignments),
            len(saved_events),
            len(saved_orders),
        ],
    }
)

validation_summary["row_count_match"] = (
    validation_summary["in_memory_rows"]
    == validation_summary["saved_file_rows"]
)

validation_summary

,table,in_memory_rows,saved_file_rows,row_count_match
0,users,20000,20000,True
1,experiment_assignments,20060,20060,True
2,events,36813,36813,True
3,orders,5043,5043,True


## Analyst Reflection

### Question 1

**Why should assignment data and event data be stored separately?**

Assignment data records which experiment group a user was allocated to, while event data records what the user actually did after assignment. Keeping them separate allows the analyst to distinguish randomization from product exposure and identify users who were assigned but never experienced the checkout flow.

### Question 2

**Why should raw duplicate and conflicting records be preserved at this stage?**

Raw issues should be preserved so that validation rules can identify and document them explicitly. Silently removing them during data generation would hide important analytical decisions and make the final dataset difficult to audit or reproduce.

### Question 3

**Why can the event table not be used directly for a two-proportion z-test?**

The event table contains multiple rows per user, while the experiment is randomized at the user level. Using event rows directly would give users with more activity greater weight and violate the intended analysis grain. The data must first be aggregated to one binary purchase outcome per eligible exposed user.

### Question 4

**Why should device and traffic source be taken from the first valid experiment exposure?**

A user may generate events from multiple devices or acquisition sources. Using the attributes associated with the first valid exposure creates one consistent segment assignment per user and prevents users from being counted in multiple segments.

### Question 5

**Why should purchases outside the 24-hour attribution window remain in the raw data?**

The raw event history should reflect what actually occurred, including late purchases. The attribution rule belongs in the analytical transformation rather than the source data, allowing the analyst to distinguish observed purchases from purchases attributed to the experiment.